# Working with APIs

Every chapter so far has run on your own machine. Chapter 10 read files from your disk, chapter 15 counted rows you had typed out yourself. This chapter is the first one that asks another computer for data.

An **API** is a way for one program to ask another program for something. When you open a weather site, your browser shows you a page; when your code wants the same weather, it calls the site's API and gets back JSON. Chapter 10 already taught you to read JSON, so the data will look familiar. The new part is getting it.

This is also the first chapter that needs something installed. Chapter 11 built a virtual environment and explained `pip` in the abstract. Here it stops being abstract.

**What we will learn:**

1. Installing `requests`, and why a `requirements.txt` now exists
2. Making a request and reading the response
3. Status codes, and the errors that are not exceptions
4. `timeout` and `raise_for_status`, the two lines people forget
5. Query parameters and headers
6. API keys, and why they never go in your code
7. Retrying, using chapter 13's decorator
8. Rate limits, and how to stay inside them
9. Pagination, using chapter 12's generators

**About the output in this notebook.** Most cells here read a saved copy of a real response from `sample_data/`, so they give the same answer every time and work with no internet. A handful make a real request to GitHub's API, and those are marked. Their committed output is from the moment the notebook was built, so your numbers will be different.

---
# 1. Installing `requests`

`requests` is not part of the standard library. Everything in chapters 1 to 16 shipped with Python; this does not.

Chapter 11 covered the whole routine. Here it is applied, in a terminal rather than a notebook:

```bash
python3 -m venv .venv               # a project-local environment
source .venv/bin/activate           # Windows: .venv\Scripts\activate
pip install requests
pip freeze > requirements.txt
```

That last line is what makes the project reproducible. This repository now has a `requirements.txt` at its root, and anyone who clones it can rebuild your environment exactly:

```bash
pip install -r requirements.txt
```

If the import below fails, that is the message telling you the environment is not set up. It is also the first time in this course that running someone else's notebook needs a step other than opening it.

In [1]:
import requests

print("requests version:", requests.__version__)

requests version: 2.32.3


Python's standard library does have `urllib.request`, and it can do everything in this chapter. Almost nobody uses it directly, because `requests` turns six lines into one. This chapter uses `requests`, which is what you will meet in real code.

---
# 2. Your First Request

`requests.get(url)` sends an HTTP GET, which is the verb for "give me this thing". It returns a **response object** holding everything the server sent back.

The next cell talks to GitHub's public API, which needs no account for a handful of requests.

In [2]:
response = requests.get("https://api.github.com/repos/python/cpython", timeout=10)

print("status code :", response.status_code)
print("ok?         :", response.ok)
print("content type:", response.headers["Content-Type"])
print("url         :", response.url)

status code : 200
ok?         : True
content type: application/json; charset=utf-8
url         : https://api.github.com/repos/python/cpython


Four things worth naming:

| | |
|---|---|
| `status_code` | a number saying how it went, covered in section 3 |
| `ok` | `True` for any status below 400 |
| `headers` | metadata the server sent, as a dictionary |
| `url` | the address actually requested, after any redirects |

The body is what you came for, and `.json()` parses it. This is chapter 10's `json.loads` with the download attached.

In [3]:
data = response.json()

print(type(data).__name__)
print("keys:", sorted(data)[:8], "...")
print()
print(data["full_name"], "|", data["description"])
print("stars:", data["stargazers_count"])

dict
keys: ['allow_forking', 'archive_url', 'archived', 'assignees_url', 'blobs_url', 'branches_url', 'clone_url', 'collaborators_url'] ...

python/cpython | The Python programming language
stars: 75972


That output is from the day this notebook was built, and the star count has moved since. From here on we use a saved copy so the numbers stop changing under us.

In [4]:
import json

with open("sample_data/github_repo.json", encoding="utf-8") as f:
    repo = json.load(f)

print(repo["full_name"], "|", repo["language"])
print("stars       :", f"{repo['stargazers_count']:,}")
print("open issues :", f"{repo['open_issues_count']:,}")
print("owner       :", repo["owner"]["login"], f"({repo['owner']['type']})")

python/cpython | Python
stars       : 75,971
open issues : 9,622
owner       : python (Organization)


`sample_data/github_repo.json` is a real response, trimmed to the fields this chapter uses and committed to the repository. Saving a response while you work on the code that handles it is a good habit in its own right: it is faster, it works on a train, and it does not spend your rate limit on the same request forty times.

### The other verbs

`GET` asks for something. HTTP has other verbs for changing things, and `requests` has a function for each:

| Verb | Means | `requests` |
|---|---|---|
| `GET` | give me this | `requests.get(url)` |
| `POST` | create something new | `requests.post(url, json=...)` |
| `PUT` | replace it entirely | `requests.put(url, json=...)` |
| `PATCH` | change part of it | `requests.patch(url, json=...)` |
| `DELETE` | remove it | `requests.delete(url)` |

`GET` is safe to repeat: asking twice gives the same answer and changes nothing. The others are not, which is why a browser warns you before resending a form.

This chapter only reads, so it only uses `GET`. But you can see exactly what a `POST` would send without sending it, by building the request and stopping there:

In [5]:
order = requests.Request("POST", "https://api.example.com/orders",
                         json={"item": "Python Crash Course", "quantity": 2},
                         headers={"Authorization": "Bearer not-a-real-key"})

prepared = order.prepare()                  # built, but never sent

print("method      :", prepared.method)
print("url         :", prepared.url)
print("content type:", prepared.headers["Content-Type"])
print("body        :", prepared.body)

method      : POST
url         : https://api.example.com/orders
content type: application/json
body        : b'{"item": "Python Crash Course", "quantity": 2}'


The `json=` argument did two things: it turned the dictionary into a JSON string with chapter 10's `json.dumps`, and it set the `Content-Type` header so the server knows how to read it. Passing `data=` instead would send it as form fields, which is what an HTML form does.

---
# 3. Status Codes

Every response carries a three-digit code. The first digit is the part to remember.

| Range | Means | You will meet |
|---|---|---|
| `2xx` | it worked | `200 OK`, `201 Created`, `204 No Content` |
| `3xx` | it moved | `301`, `302` (`requests` follows these for you) |
| `4xx` | your request was wrong | `400`, `401`, `403`, `404`, `429` |
| `5xx` | their server broke | `500`, `502`, `503` |

The `4xx` group is the one that costs people time, because the fix is different for each:

| Code | Means | Usually |
|---|---|---|
| `400 Bad Request` | the server could not read it | a malformed parameter |
| `401 Unauthorized` | no valid credentials | a missing or expired key |
| `403 Forbidden` | credentials fine, permission no | wrong scope, or a rate limit |
| `404 Not Found` | no such thing | a typo in the URL, or it was deleted |
| `429 Too Many Requests` | slow down | see section 8 |

Here is a real 404:

In [6]:
missing = requests.get("https://api.github.com/repos/python/not-a-real-repo", timeout=10)

print("status:", missing.status_code)
print("ok?   :", missing.ok)
print("body  :", missing.json()["message"])

status: 404
ok?   : False
body  : Not Found


**A failed request does not raise an exception.** That cell ran to the end without a `try` anywhere. `requests` treats "the server answered" as success, and a 404 is an answer.

This is the single most common bug in beginner API code:

```python
data = requests.get(url).json()      # a 404 body parses fine
name = data["full_name"]             # KeyError, thirty lines from the real problem
```

The 404 arrives as an ordinary dictionary, and the program carries on until something unrelated falls over.

### A 200 that is not JSON

`.json()` assumes the body is JSON. When something else answers, a login page, a proxy error, a maintenance notice, you get HTML with a perfectly healthy `200` status, and the trouble surfaces here:

In [7]:
html = "<!DOCTYPE html><html><body>Sign in to continue</body></html>"

try:
    json.loads(html)                    # this is what response.json() calls
except json.JSONDecodeError as e:
    print("JSONDecodeError:", e)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)


`response.json()` raises that same exception. When you meet it, the useful next move is `print(response.text[:200])`, because the server almost always says what went wrong in the page it sent instead.

---
# 4. The Two Lines People Forget

### `raise_for_status()`

This turns a bad status into an exception, so a failure stops where it happened. It is the bridge from HTTP into chapter 9's exception handling.

In [8]:
try:
    bad = requests.get("https://api.github.com/repos/python/not-a-real-repo", timeout=10)
    bad.raise_for_status()
    print("this line is not reached")
except requests.exceptions.HTTPError as e:
    print("HTTPError:", e)

HTTPError: 404 Client Error: Not Found for url: https://api.github.com/repos/python/not-a-real-repo


On a `2xx` it does nothing at all, so you can call it after every request without thinking about it.

### `timeout`

`requests` has **no timeout by default**. Leave it out and a request to a server that accepts your connection and then says nothing will wait forever. Not for thirty seconds: forever. A script that hangs overnight with no error message is nearly always this.

A tiny timeout shows the exception you would get:

In [9]:
try:
    requests.get("https://api.github.com/repos/python/cpython", timeout=0.001)
except requests.exceptions.Timeout as e:
    print("caught:", type(e).__name__)

caught: ConnectTimeout


The class printed is `ConnectTimeout`, which is a subclass of `Timeout`: the connection itself never opened. There is also `ReadTimeout`, for a connection that opened and then went quiet. Catching `Timeout` covers both.

`Timeout` is in turn a subclass of `requests.exceptions.RequestException`, which is the parent of everything `requests` raises. That makes it the right thing to catch when you want "the network did not work" as one case:

| Exception | Raised when |
|---|---|
| `ConnectionError` | DNS failed, connection refused, no network |
| `Timeout` | no response inside the limit you set |
| `HTTPError` | raised by `raise_for_status()` on a 4xx or 5xx |
| `RequestException` | the parent of all of them |

Put together, every request you write should look like this:

```python
try:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
except requests.exceptions.RequestException as e:
    ...
```

---
# 5. Query Parameters and Headers

### Parameters

Anything after a `?` in a URL is a query parameter. You could build that string yourself, and you should not, because values contain spaces, `&`, `=` and non-English characters that all have to be encoded.

Pass a dictionary as `params` and `requests` encodes it for you:

In [10]:
response = requests.get("https://api.github.com/search/repositories",
                        params={"q": "language:python topic:machine-learning",
                                "sort": "stars",
                                "per_page": 3},
                        timeout=10)

print("the URL that was actually sent:")
print(" ", response.url)
print()
for item in response.json()["items"]:
    print(f"  {item['full_name']:<28} {item['stargazers_count']:>7,} stars")

the URL that was actually sent:
  https://api.github.com/search/repositories?q=language%3Apython+topic%3Amachine-learning&sort=stars&per_page=3

  huggingface/transformers     164,744 stars
  pytorch/pytorch              102,740 stars
  d2l-ai/d2l-zh                 80,250 stars


Look at `response.url`. The spaces became `+`, the colons became `%3A`, and the three parameters were joined with `&`. That is what you would have had to get right by hand.

### Headers

Headers are how a request describes itself. Two matter early on:

- `User-Agent` says who is calling. Some APIs reject requests without one, and a name plus a contact address is the polite form.
- `Accept` says what format you want back.

They go in as a dictionary:

```python
headers = {"User-Agent": "sales-report/1.0 (you@example.com)",
           "Accept": "application/vnd.github+json"}

requests.get(url, headers=headers, timeout=10)
```

Response headers come back the same way, and they are worth reading. GitHub uses them to tell you about your rate limit, which section 8 covers.

---
# 6. API Keys, and Where They Do Not Go

Most APIs beyond the free tier need a key, and it usually travels in a header:

```python
headers = {"Authorization": f"Bearer {api_key}"}
```

The question is where `api_key` comes from. Not from your code:

```python
api_key = "sk-live-8f4c2a91d7e3"      # never do this
```

A key written into a file is a key you have published. It goes into git history, and deleting the line later does not remove it, because git keeps every version. It goes into the notebook's saved output. It gets shared when you send someone the file. Bots scan public repositories for exactly this pattern, and a leaked cloud key can run up a bill in hours.

The key lives in an **environment variable** instead: a value the operating system holds, outside your project.

```bash
export WEATHER_API_KEY="your-key-here"        # macOS and Linux
setx WEATHER_API_KEY "your-key-here"          # Windows
```

`os.environ` reads them, and chapter 11 already introduced `os`:

In [11]:
import os

# Standing in for a key you would have exported in your shell.
os.environ["DEMO_API_KEY"] = "sk-demo-not-a-real-key"

print("present    :", os.environ.get("DEMO_API_KEY"))
print("not present:", os.environ.get("MISSING_API_KEY"))

present    : sk-demo-not-a-real-key
not present: None


`os.environ.get()` returns `None` when the variable is not set, which is the wrong failure. The program carries on and sends `Bearer None` to the server, and the error you eventually see is a `401` that says nothing about the real cause.

Fail loudly instead, using chapter 9's `raise`:

In [12]:
def get_api_key(name):
    key = os.environ.get(name)
    if not key:
        raise RuntimeError(
            f"{name} is not set. Export it before running this script:\n"
            f'    export {name}="your-key-here"')
    return key


print("found:", get_api_key("DEMO_API_KEY")[:8] + "...")

try:
    get_api_key("MISSING_API_KEY")
except RuntimeError as e:
    print("\nRuntimeError:")
    print(e)

found: sk-demo-...

RuntimeError:
MISSING_API_KEY is not set. Export it before running this script:
    export MISSING_API_KEY="your-key-here"


Two more habits worth having from the start:

- Add `.env` and anything holding secrets to `.gitignore` on day one. This repository's `.gitignore` now has a `Secrets` section with `.env` in it, added along with this chapter, even though no secret is ever written here.
- Never print a key. The line above prints eight characters so you can see it was found, and that is as far as it should ever go.

---
# 7. Retrying, With Chapter 13's Decorator

Networks fail in a way that disks do not. A request that failed a second ago may work now, so the right response to a `ConnectionError` or a `503` is often to wait and try again.

Chapter 13 built a decorator factory for exactly this. Here it is, with the waiting doubled each time, which is called **exponential backoff**: three fast retries against a struggling server are three more problems for it.

In [13]:
import time
import functools


def retry(times=3, delay=0.5, on=Exception):
    def decorator(func):
        @functools.wraps(func)                  # chapter 13
        def wrapper(*args, **kwargs):
            wait = delay
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except on as e:
                    if attempt == times:
                        raise                   # out of tries, let it through
                    print(f"  attempt {attempt} failed ({type(e).__name__}), "
                          f"waiting {wait}s")
                    time.sleep(wait)
                    wait *= 2                   # back off
        return wrapper
    return decorator

To show it working we need a function that fails predictably, so this one fails twice and then succeeds:

In [14]:
calls = {"count": 0}


@retry(times=4, delay=0.1, on=ConnectionError)
def flaky_fetch():
    calls["count"] += 1
    if calls["count"] < 3:
        raise ConnectionError(f"call {calls['count']} refused")
    return {"status": "ok", "on_call": calls["count"]}


print("result:", flaky_fetch())

  attempt 1 failed (ConnectionError), waiting 0.1s
  attempt 2 failed (ConnectionError), waiting 0.2s


result: {'status': 'ok', 'on_call': 3}


And when it never succeeds, the exception comes through rather than being swallowed, which is the behaviour chapter 14 argued for:

In [15]:
@retry(times=3, delay=0.1, on=ConnectionError)
def always_broken():
    raise ConnectionError("host unreachable")


try:
    always_broken()
except ConnectionError as e:
    print("gave up, and said so:", e)

  attempt 1 failed (ConnectionError), waiting 0.1s
  attempt 2 failed (ConnectionError), waiting 0.2s


gave up, and said so: host unreachable


One rule about what to retry. A `503` or a timeout is worth retrying, because the request was fine and the server was not. A `404` or a `401` is not: the answer will be the same every time, and retrying a bad key just gets you blocked faster.

---
# 8. Rate Limits

An API you can call without limit is an API someone will take down by accident. Most set a budget, and tell you where you stand in the response headers.

In [16]:
limits = requests.get("https://api.github.com/rate_limit", timeout=10)
core = limits.json()["resources"]["core"]

print("allowed per hour:", core["limit"])
print("remaining       :", core["remaining"])
print("resets at       :", core["reset"], "(a Unix timestamp)")

allowed per hour: 60
remaining       : 57
resets at       : 1788461615 (a Unix timestamp)


That `reset` value is seconds since 1970, which chapter 15's `datetime` turns into something readable:

In [17]:
from datetime import datetime, timezone

reset_at = datetime.fromtimestamp(1735689600, tz=timezone.utc)
print("a reset timestamp of 1735689600 is:", reset_at.isoformat())

a reset timestamp of 1735689600 is: 2025-01-01T00:00:00+00:00


When you go over the budget you get `429 Too Many Requests`, usually with a `Retry-After` header saying how many seconds to wait. That header is an instruction, not a suggestion:

```python
if response.status_code == 429:
    wait = int(response.headers.get("Retry-After", 60))
    time.sleep(wait)
```

Three habits keep you well inside any limit:

1. **Save what you fetch.** Fetching the same thing twice in one session is the most common way to burn a budget. Chapter 10's `json.dump` is all you need.
2. **Ask for more per request.** One call returning 100 rows beats 100 calls returning one.
3. **Sleep between calls in a loop.** Even a tenth of a second changes a burst into a trickle.

The first one is worth writing down, because it is three lines:

In [18]:
from contextlib import suppress

os.makedirs("api_demo", exist_ok=True)
CACHE = "api_demo/repo.json"

with suppress(FileNotFoundError):           # chapter 14: start each run empty
    os.remove(CACHE)


def cached_json(path, fetch):
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            return json.load(f), "cache"
    data = fetch()
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)
    return data, "network"


def pretend_fetch():
    print("  (fetching over the network)")
    return {"full_name": "python/cpython", "stars": 75971}


data, source = cached_json(CACHE, pretend_fetch)
print("first call :", source)

data, source = cached_json(CACHE, pretend_fetch)
print("second call:", source, "->", data["full_name"])

  (fetching over the network)
first call : network
second call: cache -> python/cpython


`cached_json` takes the fetching function as an argument, which is chapter 7's higher-order function idea doing real work: the caching logic knows nothing about what it is caching.

The `os.remove` at the top is only so this cell tells the same story every time it runs. A real cache is the point precisely because it survives.

---
# 9. Pagination

An API will not hand you ten thousand rows at once. It gives you a page and expects you to ask for the next one.

Writing that as a loop means the caller has to know about pages. Writing it as a **generator**, which is chapter 12, means the caller just gets rows:

In [19]:
def all_items(get_page):
    # Yield items from every page, one at a time, until a page comes back empty.
    page = 1
    while True:
        items = get_page(page)
        if not items:                       # an empty page is the end
            return
        print(f"  (fetched page {page}: {len(items)} items)")
        yield from items                    # chapter 12
        page += 1

`get_page` is a function the caller supplies. That keeps the paging logic separate from where the pages come from, which is what lets us demonstrate it offline. The `print` is there so you can watch the requests happen; a real one would log instead, or say nothing.

This `get_page` reads the committed sample pages:

In [20]:
def get_page_from_samples(page):
    path = f"sample_data/issues_page_{page}.json"
    if not os.path.exists(path):
        return []
    with open(path, encoding="utf-8") as f:
        return json.load(f)


for issue in all_items(get_page_from_samples):
    print(f"    #{issue['number']} {issue['title'][:48]:<48} {issue['user']}")

  (fetched page 1: 3 items)
    #156898 [3.15] gh-89735: Document requirements for the * miss-islington
    #156897 gh-155742: Optimize json float parsing           vstinner
    #156896 Importing idlelib.run deletes tkinter dialog sub serhiy-storchaka
  (fetched page 2: 3 items)
    #156895 Configure script output doesn't mention CXX      smontanaro
    #156894 Wrong error position for tokenizer errors report serhiy-storchaka
    #156892 gh-156891: Fix shlex source inclusion at EOF     lpyu001
  (fetched page 3: 3 items)
    #156891 `shlex` silently discards a sourced stream when  lpyu001
    #156890 gh-155742: Add tests on the PyMarshal C API      vstinner
    #156889 [3.13] Reword `atexit` docs (GH-156086)          encukou


Nine issues, and four calls to `get_page`. The fourth is the one you cannot see: it returned an empty list, the generator stopped, and the `print` never ran because it sits after the check. That empty page is how an API says "there is no more".

Because it is a generator, nothing is held in memory but the page being read, and the caller can stop early. Chapter 15's `islice` takes the first four rows and the remaining pages are never fetched at all:

In [21]:
from itertools import islice

for issue in islice(all_items(get_page_from_samples), 4):
    print(f"    #{issue['number']} by {issue['user']}")

  (fetched page 1: 3 items)
    #156898 by miss-islington
    #156897 by vstinner
    #156896 by serhiy-storchaka
  (fetched page 2: 3 items)
    #156895 by smontanaro


Two pages fetched instead of four. Against a real API that is half the requests and half the wait.

The live version differs only in `get_page`:

In [22]:
def get_page_from_github(page):
    response = requests.get("https://api.github.com/repos/python/cpython/issues",
                            params={"per_page": 3, "page": page},
                            headers={"Accept": "application/vnd.github+json"},
                            timeout=10)
    response.raise_for_status()
    return response.json()


for issue in islice(all_items(get_page_from_github), 4):
    print(f"    #{issue['number']} {issue['title'][:44]}")

  (fetched page 1: 3 items)
    #156907 [3.13] gh-94242: Clarify comments for _MAX_W
    #156906 [3.14] gh-94242: Clarify comments for _MAX_W
    #156905 [3.15] gh-94242: Clarify comments for _MAX_W


  (fetched page 2: 3 items)
    #156904 [3.14] GH-146096: Fix segfault in BaseExcept


---
# 10. Putting It Together

A small client with everything this chapter argued for: a session, a timeout, headers, `raise_for_status`, and retries.

`requests.Session` is the piece we have not met. It reuses the underlying connection between calls, which is faster, and it holds headers so you set them once rather than on every request.

In [23]:
RETRYABLE = (requests.exceptions.ConnectionError,
             requests.exceptions.Timeout)


class GitHubClient:
    BASE = "https://api.github.com"

    def __init__(self, token=None, timeout=10):
        self.timeout = timeout
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/vnd.github+json",
            "User-Agent": "python-course-notebook/1.0",
        })
        if token:
            self.session.headers["Authorization"] = f"Bearer {token}"

    @retry(times=3, delay=0.5, on=RETRYABLE)
    def get(self, path, **params):
        response = self.session.get(f"{self.BASE}{path}",
                                    params=params, timeout=self.timeout)
        response.raise_for_status()
        return response.json()

    def repo(self, full_name):
        return self.get(f"/repos/{full_name}")

    def issues(self, full_name, per_page=3):
        return all_items(lambda page: self.get(f"/repos/{full_name}/issues",
                                               per_page=per_page, page=page))

    def close(self):
        self.session.close()

    def __enter__(self):                        # chapter 14
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.close()

Every method gets the timeout, the headers, the status check and the retries, because they are set in one place. Adding a token later changes one line at the call site and nothing else.

Look at what `RETRYABLE` does not include. It would have been shorter to write `on=requests.exceptions.RequestException` and catch everything, and that would have retried 404s, which section 7 said not to do: the same wrong question, asked three times, at a server that already answered it. Retrying a `503` is worth doing, and it needs more than an exception type, because `raise_for_status()` raises the same `HTTPError` for a 404 and a 503 alike. You would check `response.status_code >= 500` before deciding.

A session holds an open connection, which is exactly the acquire-and-release shape chapter 14 was about. That is why the class has `close()`, and why it has `__enter__` and `__exit__` too: a caller who uses `with` cannot forget to close it.

In [24]:
with GitHubClient() as gh:
    repo = gh.repo("python/cpython")
    print(repo["full_name"], "|", f"{repo['stargazers_count']:,} stars")

    for issue in islice(gh.issues("python/cpython"), 3):
        print(f"  #{issue['number']} {issue['title'][:44]}")

python/cpython | 75,972 stars
  (fetched page 1: 3 items)
  #156907 [3.13] gh-94242: Clarify comments for _MAX_W
  #156906 [3.14] gh-94242: Clarify comments for _MAX_W
  #156905 [3.15] gh-94242: Clarify comments for _MAX_W


Seven chapters are doing work in that one block: a class from chapter 8, exceptions from chapter 9, a generator from chapter 12, a decorator from chapter 13, a context manager from chapter 14, `islice` from chapter 15, and a lambda from chapter 7.

---
# 11. Being a Good Client

An API is someone else's server, and it costs them money to answer you.

- **Read the documentation first.** Rate limits, required headers and the shape of the response are all written down. Guessing costs more time than reading.
- **Identify yourself** with a real `User-Agent`. When something goes wrong, that is how they contact you instead of blocking you.
- **Cache aggressively.** The fastest and politest request is the one you do not make.
- **Back off when told.** Respect `Retry-After`, and treat a `429` as a message rather than an obstacle.
- **Check the terms** before you redistribute the data. Being able to fetch something is not the same as being allowed to republish it.
- **Prefer the official API** over scraping the website. It is faster, more stable, and permitted. Chapter 18 covers what to do when there is no API, and starts by telling you to look for one anyway.

---
# 12. Common Mistakes

| Mistake | What happens | Fix |
|---|---|---|
| No `timeout` | The script hangs forever with no error | `timeout=10` on every call |
| No `raise_for_status()` | A 404 body parses fine, and fails later as a `KeyError` | Check the status first |
| Key hardcoded in the file | It is published, permanently, in git history | Read it from `os.environ` |
| `os.environ.get()` with no check | `Bearer None` is sent, and you debug a 401 | Raise when the key is missing |
| Retrying a 401 or a 404 | Same answer every time, and you get blocked sooner | Only retry timeouts, 5xx and connection errors |
| Retrying with no delay | Three more requests at a server already in trouble | Back off, doubling the wait |
| Building the URL by hand | Spaces and `&` in values break it | Pass `params={...}` |
| Fetching the same thing twice | The rate limit runs out mid-job | Save the response to disk |
| Loading every page before using any | Slow, and it holds the whole dataset in memory | Yield rows with a generator |

---
# 13. Summary: Your API Cheat Sheet

**A request, written properly**

```python
import requests

try:
    response = requests.get(url,
                            params={"page": 1},
                            headers={"User-Agent": "my-app/1.0"},
                            timeout=10)
    response.raise_for_status()
    data = response.json()
except requests.exceptions.RequestException as e:
    print("request failed:", e)
```

**The response object**

| | |
|---|---|
| `response.status_code` | `200`, `404`, `500` |
| `response.ok` | `True` below 400 |
| `response.json()` | the body, parsed |
| `response.text` | the body, as a string |
| `response.headers` | a dictionary |
| `response.url` | the final URL, after encoding and redirects |

**Status codes**

| | |
|---|---|
| `2xx` | it worked |
| `3xx` | it moved |
| `4xx` | your request was wrong |
| `5xx` | their server broke |

**Exceptions**

| | |
|---|---|
| `ConnectionError` | no network, DNS failure, refused |
| `Timeout` | no answer in time |
| `HTTPError` | from `raise_for_status()` |
| `RequestException` | catch this one |

**Keys**

```bash
export WEATHER_API_KEY="..."
```

```python
key = os.environ.get("WEATHER_API_KEY")
if not key:
    raise RuntimeError("WEATHER_API_KEY is not set")
headers = {"Authorization": f"Bearer {key}"}
```

**Sessions**

```python
session = requests.Session()
session.headers.update({"User-Agent": "my-app/1.0"})
session.get(url, timeout=10)      # reuses the connection
session.close()
```

**Pagination**

```python
def all_items(get_page):
    page = 1
    while True:
        items = get_page(page)
        if not items:
            return
        yield from items
        page += 1
```

**The four lines to never leave out**

```python
timeout=10                        # or it hangs forever
response.raise_for_status()       # or a 404 becomes a KeyError
os.environ.get("API_KEY")         # or you publish your key
time.sleep(...)                   # between calls, and when told to
```

---

**Next:** chapter 18 is what you do when a site has no API. It covers BeautifulSoup, and the regex from chapter 16 that could not parse HTML. It also covers when scraping is the wrong answer, which is more often than people expect.